# Dataset Download and Compression

1. Download benchmark interaction logs (Yoochoose and Diginetica) from Kaggle.
2. Convert and store them as compressed pickle artifacts (`.pkl.gz`) in `data/raw/`.
3. Keep only columns needed for session-based recommendation preprocessing.
4. Preview metadata and sample rows.
5. Write download manifest with source names, output names, row counts, and sizes.

Requirements:
- Install python packages from `requirements.txt`.

In [ ]:
import gzip
import json
import pickle
from datetime import datetime, timezone
from pathlib import Path

import kagglehub
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def find_file(root_dir: Path, filename: str) -> Path | None:
    exact_matches = list(root_dir.rglob(filename))
    if exact_matches:
        return exact_matches[0]

    filename_lower = filename.lower()
    for path in root_dir.rglob("*"):
        if path.is_file() and path.name.lower() == filename_lower:
            return path

    return None


def write_streamed_pickle_gzip(
    source_path: Path,
    output_path: Path,
    read_csv_kwargs: dict,
    rename_columns: dict[str, str] | None,
    chunksize: int = 1_000_000,
) -> dict:
    """Read large csv/dat in chunks and write compressed pickle stream.

    File format inside `.pkl.gz`:
    1) metadata dictionary
    2) repeated DataFrame chunks
    """
    output_path.parent.mkdir(parents=True, exist_ok=True)

    chunk_iter = pd.read_csv(source_path, chunksize=chunksize, **read_csv_kwargs)
    first_chunk = next(chunk_iter)
    if rename_columns:
        first_chunk = first_chunk.rename(columns=rename_columns)

    total_rows = len(first_chunk)
    chunk_count = 1

    with gzip.open(output_path, "wb") as f:
        metadata = {
            "created_utc": datetime.now(timezone.utc).isoformat(),
            "source_file": source_path.name,
            "columns": list(first_chunk.columns),
        }
        pickle.dump(metadata, f, protocol=pickle.HIGHEST_PROTOCOL)
        pickle.dump(first_chunk, f, protocol=pickle.HIGHEST_PROTOCOL)

        for chunk in chunk_iter:
            if rename_columns:
                chunk = chunk.rename(columns=rename_columns)
            total_rows += len(chunk)
            chunk_count += 1
            pickle.dump(chunk, f, protocol=pickle.HIGHEST_PROTOCOL)

    return {
        "rows": total_rows,
        "chunks": chunk_count,
        "source_size_bytes": source_path.stat().st_size,
        "output_size_bytes": output_path.stat().st_size,
    }


DATASETS = [
    {
        "name": "diginetica",
        "kaggle_dataset": "profalbusdumbledore/diginetica-dataset",
        "tables": [
            {
                "table_name": "item_views",
                "source_filename": "train-item-views.csv",
                "output_filename": "train-item-views.pkl.gz",
                "read_csv_kwargs": {
                    "sep": ";",
                    "usecols": ["sessionId", "itemId", "timeframe", "eventdate"],
                    "dtype": {
                        "sessionId": "int64",
                        "itemId": "int64",
                        "timeframe": "int64",
                    },
                },
                "rename_columns": {
                    "sessionId": "session_id",
                    "itemId": "item_id",
                },
            }
        ],
    },
    {
        "name": "yoochoose",
        "kaggle_dataset": "phhasian0710/yoochoose",
        "tables": [
            {
                "table_name": "clicks",
                "source_filename": "yoochoose-clicks.dat",
                "output_filename": "yoochoose-clicks.pkl.gz",
                "read_csv_kwargs": {
                    "header": None,
                    "usecols": [0, 1, 2],
                    "names": ["session_id", "timestamp", "item_id"],
                    "dtype": {
                        "session_id": "int64",
                        "item_id": "int64",
                    },
                },
                "rename_columns": None,
            }
        ],
    },
]

manifest = []
PREPARED_TABLES = []

for dataset in DATASETS:
    print(f"Dataset: {dataset['name']}")
    kaggle_cache_path = Path(kagglehub.dataset_download(dataset["kaggle_dataset"]))

    raw_dataset_dir = RAW_DIR / dataset["name"]
    raw_dataset_dir.mkdir(parents=True, exist_ok=True)

    output_file_names = []

    for table in dataset["tables"]:
        source_path = find_file(kaggle_cache_path, table["source_filename"])
        output_path = None
        compression_info = None

        if source_path is None:
            print(f"Missing file in Kaggle dataset: {table['source_filename']}")
        else:
            output_path = raw_dataset_dir / table["output_filename"]
            compression_info = write_streamed_pickle_gzip(
                source_path=source_path,
                output_path=output_path,
                read_csv_kwargs=table["read_csv_kwargs"],
                rename_columns=table["rename_columns"],
            )
            rel_path = str(output_path.relative_to(RAW_DIR))
            output_file_names.append(rel_path)

        PREPARED_TABLES.append(
            {
                "dataset": dataset["name"],
                "table_name": table["table_name"],
                "source_file": table["source_filename"],
                "path": output_path,
                "compression_info": compression_info,
            }
        )

    manifest.append(
        {
            "dataset": dataset["name"],
            "kaggle_dataset": dataset["kaggle_dataset"],
            "compressed_file_names": output_file_names,
        }
    )

manifest_path = RAW_DIR / "download_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

In [ ]:
def preview_table(table_info):
    if table_info["path"] is None:
        print("\n" + "-" * 80)
        print(
            f"{table_info['dataset']} / {table_info['table_name']}: file not found after extraction."
        )
        return

    file_path = Path(table_info["path"])
    compression_info = table_info["compression_info"]

    with gzip.open(file_path, "rb") as f:
        metadata = pickle.load(f)
        first_chunk = pickle.load(f)

    source_mb = compression_info["source_size_bytes"] / 1024 / 1024
    output_mb = compression_info["output_size_bytes"] / 1024 / 1024
    ratio = (
        compression_info["output_size_bytes"] / compression_info["source_size_bytes"]
        if compression_info["source_size_bytes"]
        else 0.0
    )

    print("\n" + "-" * 80)
    print(f"Dataset: {table_info['dataset']}")
    print(f"Table: {table_info['table_name']}")
    print(f"Source file: {table_info['source_file']}")
    print(f"Rows: {compression_info['rows']:,}, chunks: {compression_info['chunks']:,}")
    print(f"Size source: {source_mb:.2f} MB")
    print(f"Size compressed: {output_mb:.2f} MB")
    print(f"Compression ratio: {ratio:.3f}")
    print(f"Fields ({len(metadata['columns'])}): {metadata['columns']}")
    print("Sample rows from first chunk:")
    print(first_chunk.head())


for table in PREPARED_TABLES:
    preview_table(table)